<div align="center">
  <hr>
  <p text-align='center'><strong>Guía para la integración de algoritmos de aprendizaje automático en EmbedIA.</strong></p>
  <img src="https://raw.githubusercontent.com/Embed-ML/EmbedIA/main/docs/assets/images/logo3.png" width=20%/>
  <hr>
</div>

Antes de comenzar esta guía repasemos brevemente y de manera simplificada el flujo de trabajo que, llegando al final de la misma, deberemos seguir:
* El usuario va a entrenar un modelo de aprendizaje automático utilizando la conocida biblioteca scikit-learn, el cual deberá guardar.
* Ese modelo con los parámetros de entrenamiento guardados será cargado y luego convertido por EmbedIA para poder realizar inferencias en microcontroladores.
<br>
<div align="center">
<img src="https://github.com/Embed-ML/EmbedIA/raw/main/docs/assets/images/workflow.png" width=60%/>
</div>

Pero...¿Cómo hace EmbedIA para convertir un modelo entrenado en scikit-learn a una implementación en C que pueda funcionar en un microcontrolador?. Para responder esta pregunta iremos paso a paso por cada una de las secciones de la <a href='#tabla-de-contenidos'>tabla de contenidos</a>. A lo largo del documento, se utilizará <strong>a modo de ejemplo</strong>, la integración del algoritmo KNeighborsClassifier (una implementación específica de K-nearest neighbors (KNN) que se utiliza para problemas de clasificación).

## Tabla de contenidos <A NAME="tabla-de-contenidos"></A>
1. [Investigación](#investigation)
2. [Desarrollo del código en C](#coding)
  - [Generación de la función init](#generation)
3. [Integrando el código en EmbedIA](#integration)
  - [Carpeta Libraries](#libraries)
  - [Carpeta Core](#core)
  - [Carpeta Wrapper](#wrappers:)
  - [Carpeta Layers](#layers)
4. [Usando EmbedIA](#usingembedia)
  - [Posibles errores](#errors)


## Investigación 🔍 <A NAME="investigation"></A>
El primer paso es investigar un algoritmo de aprendizaje automatico y entender su funcionamiento. Entrando en la página de <a href="https://scikit-learn.org/stable/index.html">scikit-learn</a>, pueden observarse varias categorías dependiendo el tipo de problema que se quiere resolver. Al elegir uno de estos, vamos a ver en la página correspondiente los parámetros de entrenamiento que posee (que es lo que más nos va a interesar en un principio), atributos y algunos ejemplos. En particular, para el algoritmo <a href="https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html#sklearn.neighbors.KNeighborsClassifier">KNeighborsClassifier</a> podemos ver que algunos de sus parámetros son ```n_neighbors```, ```weights```, entre otros.<br>
Para mejor comprensión, también es de utilidad buscar en blogs o videos, explicaciones teóricas para entender las partes más importantes del algoritmo y tener una visión más amplia del mismo.


## Desarrollo del código en C 👨‍💻 <A NAME="coding"></A>
Una vez entendidos los conceptos que rodean al algoritmo deseado, es hora de implementarlo en C. Para esto va a ser necesario crear tres archivos:
* <strong>'[nombre del algoritmo].h': </strong>Se declara la estructura del algoritmo, es decir, los parámetros de entrenamiento y otras variables que el desarrollador considere necesarios.
```c
// Ejemplo de estructura de KNeighborsClassifier en 'knn.h',
// en la que se añade únicamente el parámetro n_neighbors
typedef struct
{
    int n_neighbors;
    int n_rows;
    int n_features;
    float *neighbors_features;
    float *neighbors_labels;
} KNN;
// Cabecera de la función de predicción
int predict(KNN, float*);
```

* <strong>'[nombre del algoritmo].c': </strong>En este archivo se declaran dos funciones principales: La de predicción y la de inicialización. La primera recibe por parámetros la estructura creada anteriormente y un elemento del que se quiere inferir el resultado. <a href='#generation'>La segunda</a>, por su parte y como su nombre lo dice, inicializa todas las variables de la estrutura y retorna esta última.<br>
Además de estas dos, en este archivo se agregan todas las funciones auxiliares necesarias que va a utilizar la función de predicción.
* <strong>'main.c': </strong>Contiene al programa principal. Se crea la estructura llamando a la funcion de inicialización y luego se realiza la inferencia utilizando la funcion creada para predecir.

### Generación de la función init <A NAME="generation"></A>
Para acercarnos un poco más al funcionamiento de EmbedIA y para poder probar el código desarrollado en el paso anterior, vamos a generar el código de la función de inicialización mediante python. Para esto, vamos a crear un string al que le aplicaremos la función format(), donde cada parámetro que se le pase a esta última será reemplazado en el texto. Dentro del string, cada parámetro a reemplazar debe llevar llaves ({n_neighbors}), y como C usa llaves para definir inicios y finales, estas deben duplicarse para indicar que lo que sigue no es un parámetro.
<br>
Siguiendo con el ejemplo de KNeighborsClassifier, el string sería el siguiente:


In [ ]:
code_str = """
KNN init() {{
    static float neighbors_features[][{n_features}] = {{ {features_data} }};
    static int neighbors_labels[] = {{ {labels_data} }};

    KNN knn = {{ {n_neighbors}, {n_rows}, {n_features}, *neighbors_features, neighbors_labels }};
    return knn;
}}
"""
x_train = [1, 0.5, 3]
y_train = [0, 0, 1]

ready_code = code_str.format(
    n_neighbors=3,
    n_rows=len(y_train),
    n_features=1,
    features_data=','.join([str(x) for x in x_train]),
    labels_data=','.join([str(y) for y in y_train])
)

print(ready_code)


KNN init() {
    static float neighbors_features[][1] = { 1,0.5,3 };
    static int neighbors_labels[] = { 0,0,1 };

    KNN knn = { 3, 3, 1, *neighbors_features, neighbors_labels };
    return knn;
}



Quizá al utilizar un dataset real, copiar el código generado se vuelva engorroso, por lo que puede añadirse todo lo desarrollado en el archivo [nombre del algoritmo].c dentro del string, formatearlo al igual que la función de inicialización y luego descargarlo utilizando las siguientes líneas:
```python
with open('[nombre del algoritmo].c', 'w') as file:
  file.write(ready_code)
```
Con estas nociones y el algoritmo funcionando en C, ya podemos empezar a tocar la libería de EmbedIA 🥳

## Integración del código en EmbedIA⚙️ <A NAME="integration"></A>
Las carpetas que vamos a utilizar principalmente son: core, layers, libraries y wrappers

### Carpeta Libraries 📁 <A NAME="libraries"></A>
Dentro de esta carpeta se encuentran los algoritmos de aprendizaje automático y aprendizaje automático profundo, los cuales están agrupados según el tipo de dato de entrada y salida. Por ejemplo, entrando en 📁float podemos ver cuatro archivos:
- ```neural_net.c```: Contiene el código de los algoritmos de aprendizaje automático profundo.
- ```neural_net.h```: Contiene la estructura de los algoritmos.
- ```common.h```: Contiene la estructura de los datos de entrada y salida que vamos a usar, como puede ser **data1d_t** para datos de entrada de una dimensión de tipo flotante.
- ```common.c```

Sin embargo, dado que nosotros queremos agregar algoritmos de aprendizaje automático, no vamos a tocar ninguno de esos archivos. Por el contrario, lo que vamos a hacer es crear dos nuevos: **[nombre del algoritmo].c** y **[nombre del algoritmo.h]**. Pero...¡Estos archivos ya los tenemos! Son los que habíamos creado en la sección anterior, por lo tanto son los que vamos a agregar a la carpeta float o la correspondiente (según el tipo de datos de entrada y el de salida), con la unica diferencia que vamos a tener que cambiar el nombre de la función de predicción y ciertos tipos de datos para que coincidan con los de common.h. Entonces, para seguir con la convención de EmbedIA, aplicaremos los siguientes cambios:
- La función de predicción debe tener '_layer' al final de su nombre. Siguiendo con el ejemplo de la guía, podría llamarse 'knn_layer'. Además no devuelve nada ya que sus parámetros son la estructura, los datos de entrada y el de salida, siendo este ultimo donde se almacena el resultado.
- El nombre de la estructura deberá terminar con '_layer_t'.
- Se deberá tener en cuenta los tipos datos utilizados en 'common.h' y adaptar los demás a estos (Por ejemplo pasar los *int* a *uint*).

Siguiendo con el ejemplo de KNeighborsClassifier, el nuevo 'knn.h' quedaría asi:

```c
// Estructura del algoritmo
typedef struct
{
    uint16_t n_neighbors;
    uint32_t n_rows;
    uint16_t n_features;
    float *neighbors_features;
    float *neighbors_labels;
}k_neighbors_classifier_layer_t;
// Cabecera de la función de predicción
void k_neighbors_classifier_layer(k_neighbors_classifier_layer_t layer, data1d_t input, data1d_t * output);
```
### Carpeta Core 📁 <A NAME="core"></A>
En esta carpeta es donde hay que crear un archivo **'[nombre del algoritmo]_base_layer.py'** y definir dentro del mismo la clase **[nombre del algoritmo]BaseLayer**, similar a la definición de NeuralNetLayer de neural_net_layer.py. Esto es para definir los archivos donde estarán las implementaciones referidas a todas las variantes del algoritmo implementado(que serían [nombre del algoritmo].h y [nombre del algoritmo].c). En base al ejemplo que venimos siguiendo, el siguiente código es del archivo 'knn_base_layer.py':

```python
class KnnBaseLayer(Layer):
  def __init__(self, model, wrapper, **kwargs):
    #...
  @property
    def required_files(self):
        return super().required_files + [('knn.h', 'knn.c')]
```

### Carpeta Wrappers 📁 <A NAME="wrappers"></A>
Ya casi llegando al final de las carpetas que nos interesan, acá es donde vamos a tener que crear nuestro propio wrapper, que básicamente va a obtener los valores del modelo ya entrenado (al cual se accede mediante ```self._target```) para que podamos usarlos más adelante y adaptarlos a nuestras necesidades. Como estamos integrando modelos de aprendizaje automático, dentro de ```sklearn_wrappers.py``` vamos a agregar nuestra clase **SKL[nombre del algoritmo]Wrapper** la cual va a heredar de **ScikitLearnWrapper**. Dentro de esta, vamos a definir todas las funciones correspondientes para luego poder, por ejemplo, inicializar la estructura del algoritmo.
<br> Esto es mas fácil de ver con un ejemplo, siguiendo como de costumbre en el algoritmo KNeighborsClassifier:
```python
class SKLKnnWrapper(ScikitLearnWrapper):
    @property
    def n_neighbors(self):
        return self._target.n_neighbors
    @property
    def n_samples_fit(self):
        return self._target.n_samples_fit_
    # ...
    @property
    def activation(self):
        return None
```

Pueden definirse todas las funciones que se consideren necesarias. Sin embargo, como estamos trabajando con algoritmos que no tienen funcion de activación debemos agregarla y retornar None. También deberán definirse las funciones ```input_shape``` y ```output_shape```. En la siguiente sección vamos a utilizar varias de las que declaramos y, si aún quedan dudas, es muy probable que se aclaren ahí.

### Carpeta Layers 📁 <A NAME="layers"></A>
Por último, es acá donde va a ocurrir la inicialización, muy parecido a como habíamos hecho antes en <a href="#generation">Generación de la función init</a>, solo que en vez de usar la funcion "srt_format" para reemplazar los valores de la estructura, vamos a tilizar nuestro wrapper (al cual se accede mediante ```self._wrapper```). Dentro layers, vamos a crear una carpeta con el nombre de nuestro algoritmo y un archivo con una leyenda representativa del mismo. Por ejemplo tendríamos algo así:<br>📁knn<br>I_ 📄k_neighbors_classifier.py<br>

Para este archivo, podemos usar como base 'dense.py' ubicado en la carpeta dense. Luego las funciones que nos van a importar son
```function_implementation``` e ```invoke```. Entonces, en base a lo que hicimos anteriormente, tendremos la clase que va a heredar de **[nombre del algoritmo]BaseLayer** definido en la carpeta core y en ella las dos funciones mencionadas previamente.
- ```function_implementation```: Es donde se crea el string que vamos a usar para generar la función de inicialización tal como habíamos hecho previamente en esta guía y la retorna.
- ```invoke```: Retorna la invocación a la función que realiza la predicción **[nombre del algoritmo]_layer**, que se encuentra en el archivo realizado en la carpeta libraries.

```python
@property
def function_implementation(self):
    name = self.name
    struct_type = self.struct_data_type

    init_knn_layer = f'''
{struct_type} init_{name}_data(void){{
    uint16_t n_neighbors = {self._wrapper.n_neighbors};
    uint32_t n_rows = {self._wrapper.n_samples_fit};
    uint16_t n_features = {self._wrapper.n_features_in};
'''
    features_data = "\n".join(["{" + ", ".join(map(str, row)) + "}," for row in self._wrapper.fit_x[:-1]])
    labels_data = ','.join([str(y) for y in self._wrapper.y])
    init_knn_layer += f'''
    static float neighbors_features[][{self._wrapper.n_features_in}] = {features_data};
    static float neighbors_labels[] = {labels_data};
'''
    init_knn_layer += f'''
    k_neighbors_classifier_layer_t layer= {{ n_neighbors, n_rows, n_features, *neighbors_features, neighbors_labels}};
    return layer;
}}
'''
    return init_knn_layer

def invoke(self, input_name, output_name):
    return f'''k_neighbors_classifier_layer({self.name}_data, {input_name}, &{output_name});'''
```

Para ir terminando, debemos volver por un instante a la carpeta 📁core, ya que si bien definimos el wrapper aún no lo asociamos con el algoritmo de scikit-learn correspondiente para que pueda obtener los valores de entrenamiento. Entonces, en esa carpeta, ingresamos al archivo ```layers_implemented.py``` en el cual vamos a importar el algoritmo de scikit-learn, nuestro wrapper y nuestra clase de la carpeta 📁layers, y dentro de 'dict_layers' vamos a asociarlos:

```python
from sklearn import neighbors
#...
dict_layers = {
#...
neighbors.KNeighborsClassifier: (KNeighborsClassifier, SKLKnnWrapper)}
```
Con esto ya podríamos probar si la generación del programa funciona...¡A cruzar los dedos! 🤞

## Usando EmbedIA 🤖 <A NAME="usingembedia"></A>
Para generar entonces el programa, vamos a utilizar el notebook ```using_EmbedIA.ipynb``` y seguir la mayoría de pasos del mismo:
- Instalar larq.
- Ubicarse dentro de la carpeta del proyecto ```%cd EmbedIA``` (o el nombre de la carpeta en el que se encuentre).
- Cargar el dataset.
- Crear el modelo de aprendizaje automático y entrenarlo.
- Guardar el modelo entrenado utilizando alguna librería como 'pickle' o 'joblib' dentro de la carpeta ```models```
- Convertir el modelo para la inferencia en el microcontrolador. (En esta sección hay que hacer un pequeño cambio, ya que como guardamos el modelo con pickle o joblib, debemos cargarlo de la misma manera. Entonces en vez de usar ```model = load_model(MODEL_FILE)``` de tensorflow vamos a poner ```model = joblib.load(MODEL_FILE)``` en el caso de joblib por ejemplo)

Si todo salió bien deberíamos recibir un resultado como este:

```
+---------------------+-------------------+------------+--------+------+------------+
| EmbedIA Layer       | Name              | #Param(NT) | Shape  | MACs | Size (KiB) |
+---------------------+-------------------+------------+--------+------+------------+
| KNeighborsClassifier| s_k_l_knn_wrapper |          0 | (353,) |    0 |     0.000  |
+---------------------+-------------------+------------+--------+------+------------+
Total params (NT)....: 0
Total size in KiB....: 0.000
Total MACs operations: 0

Project knn_project exported in outputs/
```

En caso de no aparecer así, a continuación se listan algunos posibles errores al ejecutar la celda de la conversión del modelo:

### Posibles errores⚠️ <A NAME="errors"></A>
- **AttributeError: module 'keras._tf_keras.keras.layers' has no attribute 'LocallyConnected1D'**: Se puede solucionar bajando el nivel de la version de tensorflow (Por ejemplo a 2.14).
- **'[algoritmo]' object has no attribute 'name'**: En la funcion generate_embedia_model(..., model.name,...) en vez de model.name se puede reemplazar por un string.


Felicidades 🎉, integraste un nuevo algoritmo en EmbedIA!!
Para poder descargar todos los archivos que se generaron, pueden usarse estas lineas de código:
```python
from google.colab import files

!zip -r embedia_project.zip 'outputs/knn_project'
files.download('/content/EmbedIA/embedia_project.zip')
```

<small>(Es posible que en main.c se haya generado un #include "neural_net.h", en ese caso borrar esa línea.)</small>

### Muchas gracias por llegar al final de la guía, espero que te haya sido útil! 😄👋